#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col
from pyspark.sql.window import Window

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

##Checking Bronze Table

In [0]:
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


#Data Transformations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Product Key Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

##Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

##Product Line Normalization

In [0]:
df = (
    df
    # Normalize product line
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

##Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

##Calculate end date as one day before the next start date

In [0]:
window_spec = Window.partitionBy('prd_key').orderBy(col('prd_start_dt'))
next_start_date = F.lead('prd_start_dt').over(window_spec)
df = df.withColumn('prd_end_dt', F.date_sub(next_start_date, 1).cast(DateType()))

##Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks of dataframe

In [0]:
df.limit(10).display()

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28


#Write Into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products
LIMIT 10

product_id,product_number,product_name,product_cost,product_line,start_date,end_date,category_id
601,BB-7421,LL Bottom Bracket,24,n/a,2013-07-01,null,CO_BB
602,BB-8107,ML Bottom Bracket,45,n/a,2013-07-01,null,CO_BB
603,BB-9108,HL Bottom Bracket,54,n/a,2013-07-01,null,CO_BB
478,BC-M005,Mountain Bottle Cage,4,Mountain,2013-07-01,null,AC_BC
479,BC-R205,Road Bottle Cage,3,Road,2013-07-01,null,AC_BC
596,BK-M18B-40,Mountain-500 Black- 40,295,Mountain,2013-07-01,null,BI_MB
597,BK-M18B-42,Mountain-500 Black- 42,295,Mountain,2013-07-01,null,BI_MB
598,BK-M18B-44,Mountain-500 Black- 44,295,Mountain,2013-07-01,null,BI_MB
599,BK-M18B-48,Mountain-500 Black- 48,295,Mountain,2013-07-01,null,BI_MB
600,BK-M18B-52,Mountain-500 Black- 52,295,Mountain,2013-07-01,null,BI_MB
